# 9. Paper 1 model tables and plots

Assemble publication outputs for the stacked conditional-logit model and the XGBoost/neural winners frozen in Stage 8. Stage 9 validates the SHA-256 hashes of the development-CV summary, experiment plan, and frozen ranking before it opens any held-out artifact. It never reselects a model from held-out performance.

Primary outputs use `all`. Set `COMMENTGAP_MODEL_SCOPES=all,root` to add matching appendix tables. The reporter refuses to run while the factorial launcher is active.

In [ ]:
from pathlib import Path
import json
import os

import pandas as pd
from IPython.display import display
from commentgap_analysis.explanations import summarize_shap_story_variability, summarize_shap_values

from commentgap_analysis.paper1_reporting import run_paper1_reporting
from commentgap_analysis.paper1_plotting import (
    plot_regression_selector_coefficients,
    plot_regression_selector_differences,
    plot_regression_vs_shap_gaps,
    plot_winner_permutation_importance,
    plot_winner_permutation_importance_gaps,
    plot_winner_shap_importance,
    plot_winner_shap_importance_gaps,
)

scope_text = os.getenv("COMMENTGAP_MODEL_SCOPES", "all")
SCOPES = tuple(dict.fromkeys(part.strip() for part in scope_text.split(",") if part.strip()))
if not SCOPES or not set(SCOPES).issubset({"all", "root"}):
    raise ValueError(f"Invalid COMMENTGAP_MODEL_SCOPES={scope_text!r}")
MODEL_DATA_ROOT = Path(os.getenv("COMMENTGAP_MODEL_DATA_ROOT", "model_output/selection_2025/model_data"))
FACTORIAL_ROOT = Path(os.getenv("COMMENTGAP_FACTORIAL_ROOT", "model_output/selection_2025/factorial_rankers"))
WINNER_ROOT = Path(os.getenv("COMMENTGAP_FACTORIAL_WINNER_ROOT", "model_output/selection_2025/paper1/factorial_winners"))
REGRESSION_ROOT = Path(os.getenv("COMMENTGAP_REGRESSION_ROOT", "model_output/selection_2025/regression"))
OUTPUT_ROOT = Path(os.getenv("COMMENTGAP_REPORT_ROOT", "model_output/selection_2025/paper1/reporting"))
TABLES = OUTPUT_ROOT / "tables"
PERMUTATION_REPEATS = int(os.getenv("COMMENTGAP_PERMUTATION_REPEATS", "1"))
SHAP_TEST_ROWS = int(os.getenv("COMMENTGAP_SHAP_TEST_ROWS", "50000"))
SHAP_BACKGROUND_ROWS = int(os.getenv("COMMENTGAP_SHAP_BACKGROUND_ROWS", "2048"))
SHAP_NSAMPLES = int(os.getenv("COMMENTGAP_SHAP_NSAMPLES", "100"))
SHAP_FORCE_RECOMPUTE = os.getenv("COMMENTGAP_SHAP_FORCE_RECOMPUTE", "0") == "1"
SHAP_CHUNK_ROWS = int(os.getenv("COMMENTGAP_SHAP_CHUNK_ROWS", "500"))
{"scopes": SCOPES, "winner_manifest": str(WINNER_ROOT / "factorial_winner_manifest.json"), "output": str(OUTPUT_ROOT)}

In [ ]:
report_manifest = run_paper1_reporting(
    model_data_root=MODEL_DATA_ROOT,
    factorial_root=FACTORIAL_ROOT,
    winner_root=WINNER_ROOT,
    regression_root=REGRESSION_ROOT,
    output_root=OUTPUT_ROOT,
    scopes=SCOPES,
    bootstrap_draws=1000,
    permutation_repeats=PERMUTATION_REPEATS,
    shap_test_rows=SHAP_TEST_ROWS,
    shap_background_rows=SHAP_BACKGROUND_ROWS,
    shap_nsamples=SHAP_NSAMPLES,
    shap_force_recompute=SHAP_FORCE_RECOMPUTE,
    shap_chunk_rows=SHAP_CHUNK_ROWS,
    make_figures=True,
    make_latex_tables=False,
)
report_manifest

## Selector feature gaps

This section compares three complementary views of feature use: stacked-model coefficients, held-out permutation importance, and SHAP attribution. The regression feature gap is the signed curator-minus-audience interaction on the log-odds scale. XGBoost and neural feature gaps are paired within-article permutation losses: curator nDCG@k importance minus audience nDCG@k importance. SHAP plots use signed mean attribution and signed editor-minus-audience gaps. Positive values indicate an upward contribution or greater editor-specific contribution; negative values indicate a downward contribution or greater audience-specific contribution. Audience importance averages the ten deterministic tie draws. Permutations cover the named tabular features; frozen BGE vectors remain fixed rather than treating their 1,024 latent dimensions as separate substantive features.

In [ ]:
regression_feature_gaps = pd.read_csv(TABLES / "regression_feature_gaps.csv")
permutation_gaps = pd.read_csv(TABLES / "held_out_permutation_importance_gaps.csv")
display(regression_feature_gaps)
display(permutation_gaps.sort_values("permutation_importance_gap", key=abs, ascending=False))
display(plot_regression_selector_coefficients(regression_feature_gaps, output_root=None, show=False))
display(plot_regression_selector_differences(regression_feature_gaps, output_root=None, show=False))
display(plot_winner_permutation_importance(permutation_gaps, output_root=None, show=False))
display(plot_winner_permutation_importance_gaps(permutation_gaps, output_root=None, show=False))

## SHAP artifacts

Stage 9 calculates SHAP for the frozen XGBoost and neural winners on the sealed `paper2_test` comments. The background is development-only; BGE dimensions are summed into the `text_bge` block. The report exports both selector-specific signed mean SHAP contributions and signed editor-minus-audience SHAP gaps; the figures show the story-level interquartile range (IQR) as variability bars, while mean absolute SHAP remains available in the table as an importance diagnostic. Notebook 12 consumes the story-aggregated summary for the FORUM comparison.

In [ ]:
shap_values = pd.read_parquet(TABLES / "held_out_shap_values.parquet")
shap_summary = summarize_shap_values(shap_values)
shap_story_variability = summarize_shap_story_variability(shap_values)
shap_summary = shap_summary.merge(
    shap_story_variability,
    on=["scope", "model_family", "model_id", "feature_set", "feature"],
    how="left", validate="one_to_one",
)
shap_summary.to_parquet(TABLES / "held_out_shap_importance.parquet", index=False)
shap_summary.to_csv(TABLES / "held_out_shap_importance.csv", index=False)
shap_story_variability.to_parquet(TABLES / "held_out_shap_story_variability.parquet", index=False)
display(shap_summary.head(20))
print(f"{len(shap_summary):,} story-aggregated SHAP rows")
display(plot_winner_shap_importance(shap_summary, output_root=OUTPUT_ROOT, show=False))
for shap_gap_figure in plot_winner_shap_importance_gaps(shap_summary, output_root=OUTPUT_ROOT, show=False, limit=40):
    display(shap_gap_figure)
display(plot_regression_vs_shap_gaps(regression_feature_gaps, shap_summary, output_root=OUTPUT_ROOT, show=False))

## Frozen winners and development CV

In [ ]:
TABLES = OUTPUT_ROOT / "tables"
winners = pd.read_csv(TABLES / "development_cv_winners.csv")
cv_ranking = pd.read_csv(TABLES / "development_cv_variant_ranking.csv")
display(winners)
display(cv_ranking.groupby(["family", "scope"], group_keys=False).head(10))

## Development and held-out results

The first table is in-sample development performance from the frozen models fitted on all development articles; it has the same audience/editor nDCG and balanced macro-F1 columns as the sealed test table. These values are descriptive training-set diagnostics, not estimates of generalisation. The separate development-CV table above remains the model-selection result. The second table is the sealed `paper2_test` evaluation and is the source of Figure 26.

These results are opened only after winner freezing. Paired differences are clustered by held-out article and use the second-minus-first direction named in `comparison`. For `mean_selected_rank`, lower values are better; the other reported ranking metrics are better when higher. The balanced macro-F1 table is a secondary paper-comparison diagnostic: non-selected comments are undersampled within each article-selector query, the top k scores are selected, and macro-F1 is averaged over articles.

In [ ]:
import numpy as np

from commentgap_analysis.neural_ranking import scores_to_long
from commentgap_analysis.paper1_reporting import (
    _bootstrap_group_mean,
    _selector_article_ndcg,
    balanced_macro_f1_at_k,
    score_development_model,
)

development = pd.read_csv(TABLES / "development_cv_winners.csv")
performance = pd.read_csv(TABLES / "held_out_model_performance.csv")
paired = pd.read_csv(TABLES / "held_out_paired_model_differences.csv")
balanced_f1 = pd.read_csv(TABLES / "held_out_balanced_macro_f1.csv")
balanced_f1_paired_path = TABLES / "held_out_paired_balanced_macro_f1.csv"
balanced_f1_paired = pd.read_csv(balanced_f1_paired_path) if balanced_f1_paired_path.exists() else pd.DataFrame()

model_order = {"Reg": 0, "XGB": 1, "XGB-T": 2, "NN": 3, "NN-T": 4}
development_labels = {
    ("xgboost", "metadata"): "XGB",
    ("xgboost", "metadata_bge"): "XGB-T",
    ("neural", "metadata"): "NN",
    ("neural", "metadata_bge"): "NN-T",
}
development = development[development["scope"].eq("all")].copy()
development["Model"] = [
    development_labels.get((family, feature_set))
    for family, feature_set in zip(development["family"], development["feature_set"])
]
winner_records = {
    str(row["variant_id"]): row.to_dict()
    for _, row in development.loc[development["scope"].eq("all")].iterrows()
}
training_article_frames = []
for index, label in enumerate(("Reg", "XGB", "XGB-T", "NN", "NN-T")):
    if label == "Reg":
        wide = score_development_model(
            model_family="conditional_logit",
            model_data_root=MODEL_DATA_ROOT,
            factorial_root=FACTORIAL_ROOT,
            regression_root=REGRESSION_ROOT,
            scope="all",
        )
    else:
        winner = next(
            record for record in winner_records.values()
            if development_labels.get((record["family"], record["feature_set"])) == label
        )
        wide = score_development_model(
            model_family=str(winner["family"]),
            model_data_root=MODEL_DATA_ROOT,
            factorial_root=FACTORIAL_ROOT,
            regression_root=REGRESSION_ROOT,
            winner=winner,
            scope="all",
        )
    ndcg_articles = _selector_article_ndcg(wide)
    f1_articles = balanced_macro_f1_at_k(
        scores_to_long(wide, audience_draw=1),
        draws=100,
        seed=20260813 + index * 100_000,
    )
    article = ndcg_articles.merge(
        f1_articles[["story_id", "selector", "balanced_macro_f1_at_k"]],
        on=["story_id", "selector"],
        validate="one_to_one",
    )
    training_article_frames.append(article.assign(Model=label))

training_articles = pd.concat(training_article_frames, ignore_index=True)
training_metric_frames = []
for metric in ("ndcg_at_k", "balanced_macro_f1_at_k"):
    summary = _bootstrap_group_mean(
        training_articles,
        groups=["Model", "selector"],
        value=metric,
        draws=1000,
        seed=20260813 + (0 if metric == "ndcg_at_k" else 1_000_000),
        estimate_name="estimate",
    )
    summary["metric"] = metric
    training_metric_frames.append(summary)
training_metrics = pd.concat(training_metric_frames, ignore_index=True)

def _training_ci(frame, label, selector, metric):
    row = frame.loc[
        frame["Model"].eq(label)
        & frame["selector"].eq(selector)
        & frame["metric"].eq(metric)
    ].iloc[0]
    return f"{row['estimate']:.3f} [{row['conf_low']:.3f}, {row['conf_high']:.3f}]"

training_table = pd.DataFrame([
    {
        "Model": label,
        "Audience nDCG@k": _training_ci(training_metrics, label, "audience", "ndcg_at_k"),
        "Editor nDCG@k": _training_ci(training_metrics, label, "curator", "ndcg_at_k"),
        "Audience macro-F1": _training_ci(training_metrics, label, "audience", "balanced_macro_f1_at_k"),
        "Editor macro-F1": _training_ci(training_metrics, label, "curator", "balanced_macro_f1_at_k"),
    }
    for label in ("Reg", "XGB", "XGB-T", "NN", "NN-T")
])
display(training_table)
training_table.to_csv(TABLES / "figure_26_training_model_performance.csv", index=False)
def _bold_best_latex(frame, columns, best_labels=None):
    output = frame.copy()
    for column in columns:
        if best_labels is None:
            estimates = pd.to_numeric(
                output[column].astype(str).str.extract(r"^\s*([-+]?[0-9]*\.?[0-9]+)")[0],
                errors="coerce",
            )
            best_rows = estimates.eq(estimates.max())
        else:
            best_rows = output["Model"].isin(best_labels[column])
        output.loc[best_rows, column] = output.loc[
            best_rows, column
        ].map(lambda value: rf"\textbf{{{value}}}")
    return output
training_columns = ["Audience nDCG@k", "Editor nDCG@k", "Audience macro-F1", "Editor macro-F1"]
training_latex = _bold_best_latex(training_table, training_columns)
training_latex.to_latex(TABLES / "figure_26_training_model_performance.tex", index=False, escape=False)
def _performance_ci(frame, label, selector, metric):
    row = frame.loc[
        frame["model_pair_label"].eq(label)
        & frame["selector"].eq(selector)
        & frame["metric"].eq(metric)
    ].iloc[0]
    return f"{row['estimate']:.3f} [{row['conf_low']:.3f}, {row['conf_high']:.3f}]"

performance_table = (
    performance.assign(order=performance["model_pair_label"].map(model_order))
    .sort_values(["order", "selector", "metric"])
)
labels = performance_table["model_pair_label"].drop_duplicates().tolist()
table_rows = []
for label in labels:
    table_rows.append({
        "Model": label,
        "Audience nDCG@k": _performance_ci(performance_table, label, "audience", "ndcg_at_k"),
        "Editor nDCG@k": _performance_ci(performance_table, label, "curator", "ndcg_at_k"),
        "Audience macro-F1": _performance_ci(balanced_f1, label, "audience", "balanced_macro_f1_at_k"),
        "Editor macro-F1": _performance_ci(balanced_f1, label, "curator", "balanced_macro_f1_at_k"),
    })
test_table = pd.DataFrame(table_rows)
display(test_table)
test_table.to_csv(TABLES / "figure_26_test_model_performance.csv", index=False)
test_columns = ["Audience nDCG@k", "Editor nDCG@k", "Audience macro-F1", "Editor macro-F1"]
test_sources = {
    "Audience nDCG@k": (performance, "audience", "ndcg_at_k"),
    "Editor nDCG@k": (performance, "curator", "ndcg_at_k"),
    "Audience macro-F1": (balanced_f1, "audience", "balanced_macro_f1_at_k"),
    "Editor macro-F1": (balanced_f1, "curator", "balanced_macro_f1_at_k"),
}
test_best_labels = {}
for column, (source, selector, metric) in test_sources.items():
    subset = source[source["selector"].eq(selector) & source["metric"].eq(metric)]
    best_estimate = subset["estimate"].max()
    test_best_labels[column] = subset.loc[subset["estimate"].eq(best_estimate), "model_pair_label"].tolist()
test_latex = _bold_best_latex(test_table, test_columns, best_labels=test_best_labels)
test_latex.to_latex(TABLES / "figure_26_test_model_performance.tex", index=False, escape=False)
# Backward-compatible Figure 26 table name: this is the held-out test table.
test_table.to_csv(TABLES / "figure_26_model_performance.csv", index=False)
test_latex.to_latex(TABLES / "figure_26_model_performance.tex", index=False, escape=False)
display(performance)
display(paired[paired["metric"] == "ndcg_at_k"] )
display(balanced_f1)
if not balanced_f1_paired.empty:
    display(balanced_f1_paired)

## Regression associations and tie sensitivity

In [ ]:
associations = pd.read_csv(TABLES / "regression_selector_associations.csv")
tie_sensitivity = pd.read_csv(TABLES / "held_out_tie_sensitivity.csv")
display(associations)
display(tie_sensitivity)
json.loads((OUTPUT_ROOT / "report_manifest.json").read_text())

## Full regression table (LaTeX and rendered HTML)

In [ ]:
import re
import numpy as np
from html import escape as html_escape
from IPython.display import HTML, display

def _safe_re_sub(pattern, replacement, text, **kwargs):
    return re.sub(pattern, lambda _: replacement, text, **kwargs)

REGRESSION_TABLE_DIR = OUTPUT_ROOT / "tables"
REGRESSION_TABLE_DIR.mkdir(parents=True, exist_ok=True)
COLLINEARITY_ROOT = Path(os.getenv("COMMENTGAP_COLLINEARITY_ROOT", str(REGRESSION_ROOT.parent / "regression_sensitivity" / "collinearity_shared_preprocessing_v4")))

def _clean_regression_label(value):
    return (str(value).replace(" (raw expected ordinal score)", "").replace(" at the development-reply mean", " at the mean").replace("Overnight posting period", "Overnight posting period vs workday").replace("Weekday shoulder/evening", "Weekday shoulder/evening vs workday").replace("Weekend daytime/evening", "Weekend daytime/evening vs workday").removeprefix("AQuA "))

def _regression_feature_group(term):
    if str(term).startswith("aqua_"):
        return "aqua_expected"
    if str(term) in {"log_words", "sentiment_positive", "sentiment_negative",
                     "toxicity_probability", "lexdiv_length_adjusted",
                     "reading_level_length_adjusted", "url_present"}:
        return "text_nlp"
    if str(term).startswith("log_author_") or str(term).startswith("author_prior_"):
        return "author_history"
    if str(term) in {"is_reply", "reply_depth_centered", "prior_reply_composition"}:
        return "reply_structure"
    return "semantic_timing_activity"
FEATURE_GROUP_LABELS = {
    "text_nlp": "Text and language",
    "semantic_timing_activity": "Discussion context and timing",
    "author_history": "Author history",
    "reply_structure": "Reply structure",
    "aqua_expected": "AQuA dimensions",
}

def _fmt_estimate(estimate, low, high):
    return f"{estimate:.3f} [{low:.3f}, {high:.3f}]"

def _fmt_p_value(value):
    if pd.isna(value):
        return "--"
    return "<0.001" if value < 0.001 else f"{value:.3f}"

def _as_number(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def _fmt_diagnostic(key, value):
    if value is None or pd.isna(value):
        return "--"
    text = str(value)
    if text in {"TRUE", "True"}:
        return "yes"
    if text in {"FALSE", "False"}:
        return "no"
    number = _as_number(value)
    if number is None:
        return text
    if key.endswith("p_value"):
        return _fmt_p_value(number)
    if key in {"n_observations", "n_events", "n_parameters", "n_strata",
               "development_articles", "paper2_test_articles",
               "development_candidate_comments", "paper2_test_candidate_comments",
               "development_curator_selections", "development_audience_selections",
               "development_events_total", "development_stacked_rows",
               "tie_draws_fitted", "article_selector_strata",
               "n_model_parameters", "iterations", "likelihood_ratio_df",
               "model_wald_df", "robust_score_df"}:
        return f"{int(number):,}"
    if key in {"concordance", "concordance_se"}:
        return f"{number:.3f}"
    return f"{number:,.2f}"

def _heldout_regression_metrics(scores):
    """Calculate test-set pairwise concordance and exact fixed-k set log loss."""
    rows = []
    for (story_id, selector), frame in scores.groupby(["story_id", "selector"], sort=False):
        values = frame["score"].to_numpy(dtype=float)
        selected = frame["selected"].to_numpy(dtype=bool)
        selected_scores = values[selected]
        unselected_scores = np.sort(values[~selected])
        n_pairs = len(selected_scores) * len(unselected_scores)
        if n_pairs:
            below = np.searchsorted(unselected_scores, selected_scores, side="left")
            tied = np.searchsorted(unselected_scores, selected_scores, side="right") - below
            concordant_pairs = float(np.sum(below + 0.5 * tied))
        else:
            concordant_pairs = np.nan
        k = int(frame["n_picks"].iloc[0])
        log_elementary = np.full(k + 1, -np.inf)
        log_elementary[0] = 0.0
        seen = 0
        for score in values:
            upper = min(k, seen + 1)
            for degree in range(upper, 0, -1):
                log_elementary[degree] = np.logaddexp(
                    log_elementary[degree], log_elementary[degree - 1] + score
                )
            seen += 1
        conditional_log_loss = -(float(values[selected].sum()) - log_elementary[k])
        rows.append({
            "story_id": story_id,
            "selector": selector,
            "concordant_pairs": concordant_pairs,
            "comparable_pairs": n_pairs,
            "conditional_log_loss": conditional_log_loss,
        })
    return pd.DataFrame(rows)

def _heldout_metric_summary(stratum_metrics, draws=1000, seed=90210):
    story_metrics = stratum_metrics.groupby("story_id", as_index=False).agg(
        concordant_pairs=("concordant_pairs", "sum"),
        comparable_pairs=("comparable_pairs", "sum"),
        conditional_log_loss=("conditional_log_loss", "mean"),
    )
    estimate_concordance = story_metrics["concordant_pairs"].sum() / story_metrics["comparable_pairs"].sum()
    estimate_log_loss = story_metrics["conditional_log_loss"].mean()
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(story_metrics), size=(draws, len(story_metrics)))
    sampled = story_metrics.iloc[indices.ravel()][["concordant_pairs", "comparable_pairs", "conditional_log_loss"]].to_numpy(dtype=float).reshape(draws, len(story_metrics), -1)
    concordance_draws = sampled[:, :, 0].sum(axis=1) / sampled[:, :, 1].sum(axis=1)
    log_loss_draws = sampled[:, :, 2].mean(axis=1)
    return {
        "heldout_concordance": f"{estimate_concordance:.3f} [{np.quantile(concordance_draws, 0.025):.3f}, {np.quantile(concordance_draws, 0.975):.3f}]",
        "heldout_conditional_log_loss": f"{estimate_log_loss:.2f} [{np.quantile(log_loss_draws, 0.025):.2f}, {np.quantile(log_loss_draws, 0.975):.2f}]",
    }

diagnostic_order = [
    ("development_articles", "Development articles"),
    ("paper2_test_articles", "Held-out articles"),
    ("development_candidate_comments", "Development candidate comments"),
    ("paper2_test_candidate_comments", "Held-out candidate comments"),
    ("development_curator_selections", "Development editor selections"),
    ("development_audience_selections", "Development audience selections"),
    ("development_events_total", "Development selected events"),
    ("n_observations", "Stacked observations"),
    ("n_events", "Stacked selected events"),
    ("n_strata", "Article-selector strata"),
    ("n_parameters", "Substantive parameters"),
    ("iterations", "Newton-Raphson iterations"),
    ("null_log_likelihood", "Null log likelihood"),
    ("fitted_log_likelihood", "Fitted log likelihood"),
    ("aic", "AIC"),
    ("bic", "BIC"),
    ("likelihood_ratio_chisq", "Likelihood-ratio chi-square"),
    ("likelihood_ratio_df", "Likelihood-ratio degrees of freedom"),
    ("likelihood_ratio_p_value", "Likelihood-ratio test p"),
    ("model_wald_chisq", "Model-based Wald chi-square"),
    ("model_wald_df", "Model-based Wald degrees of freedom"),
    ("model_wald_p_value", "Model-based Wald test p"),
    ("robust_score_chisq", "Cluster-robust score chi-square"),
    ("robust_score_df", "Cluster-robust score degrees of freedom"),
    ("robust_score_p_value", "Cluster-robust score test p"),
    ("concordance", "Concordance"),
    ("concordance_se", "Concordance SE"),
    ("maximum_clustered_standard_error", "Maximum clustered SE"),
    ("correlation_scale_condition_number", "Robust correlation-scale condition number"),
    ("maximum_condition_index", "Maximum robust condition index"),
    ("heldout_concordance", "Held-out concordance [95% CI]"),
    ("heldout_conditional_log_loss", "Held-out conditional log loss [95% CI]"),
    ("fit_warning_count", "Fit warning count"),
    ("all_coefficients_finite", "All coefficients finite"),
    ("all_standard_errors_finite", "All standard errors finite"),
]

scope_coefficient_tables = {}
scope_diagnostic_tables = {}
missing_regression_numbers = []
latex_sections = [
    "% Requires \\usepackage{booktabs,longtable}",
    "% Standard errors are clustered by article; coefficients are on the",
    "% common conditional-logit log-odds scale.",
    "\\setlength{\\tabcolsep}{2pt}",
]
html_sections = [
    "<style>.regression-table thead tr:first-child th:nth-child(2),.regression-table thead tr:first-child th:nth-child(3),.regression-table thead tr:first-child th:nth-child(4),.regression-table thead tr:nth-child(2) th:nth-child(3),.regression-table thead tr:nth-child(2) th:nth-child(6),.regression-table thead tr:nth-child(2) th:nth-child(9),.regression-table tbody td:nth-child(3),.regression-table tbody td:nth-child(6),.regression-table tbody td:nth-child(9){border-left:2px solid #555;border-right:1px solid #bbb}</style>",
    "<style>.regression-table thead tr:first-child th:last-child{border-right:0}</style>",
]
all_coefficient_rows = []
all_diagnostic_rows = []

for scope in ("all",):
    scope_root = REGRESSION_ROOT / scope
    associations_scope = pd.read_csv(scope_root / "selector_associations.csv")
    diagnostics_scope = pd.read_csv(scope_root / "model_diagnostics.csv", dtype=str)
    sample_scope = pd.read_csv(scope_root / "sample_summary.csv", dtype=str)
    diagnostics = dict(zip(diagnostics_scope["statistic"], diagnostics_scope["value"]))
    sample = dict(zip(sample_scope["statistic"], sample_scope["value"]))
    heldout_stratum_metrics = _heldout_regression_metrics(pd.read_parquet(scope_root / "test_scores_long.parquet"))
    diagnostic_overrides = _heldout_metric_summary(heldout_stratum_metrics)
    conditioning_path = COLLINEARITY_ROOT / scope / "covariance_conditioning.csv"
    if conditioning_path.exists():
        conditioning = pd.read_csv(conditioning_path)
        robust_conditioning = conditioning.loc[conditioning["covariance"].eq("cluster_robust")].iloc[0]
        diagnostic_overrides.update({
            "correlation_scale_condition_number": str(robust_conditioning["correlation_scale_condition_number"]),
            "maximum_condition_index": str(robust_conditioning["corresponding_max_condition_index"]),
        })
    feature_group_order = {name: index for index, name in enumerate(("text_nlp", "semantic_timing_activity", "author_history", "reply_structure", "aqua_expected"))}
    associations_scope = associations_scope.copy()
    associations_scope["_feature_group"] = associations_scope["term"].map(_regression_feature_group)
    associations_scope["_feature_group_order"] = associations_scope["_feature_group"].map(feature_group_order)
    associations_scope = associations_scope.sort_values("_feature_group_order", kind="stable").drop(columns=["_feature_group", "_feature_group_order"])

    coefficient_table_flat = pd.DataFrame({
        "Feature": associations_scope["feature"].map(_clean_regression_label),
        "Unit": associations_scope["unit"].replace({"One standard deviation": "1 SD", "Absent to present": "1 vs 0"}),
        "Audience beta [95% CI]": [
            _fmt_estimate(row.audience_log_odds, row.audience_conf_low, row.audience_conf_high)
            for row in associations_scope.itertuples()
        ],
        "Audience p": associations_scope["audience_p_value"].map(_fmt_p_value),
        "Audience q": associations_scope["audience_q_value_bh"].map(_fmt_p_value),
        "Editor beta [95% CI]": [
            _fmt_estimate(row.curator_log_odds, row.curator_conf_low, row.curator_conf_high)
            for row in associations_scope.itertuples()
        ],
        "Editor $p$": associations_scope["curator_p_value"].map(_fmt_p_value),
        "Editor $q$": associations_scope["curator_q_value_bh"].map(_fmt_p_value),
        "Editor $-$ audience $\\Delta\\beta$ [95\\% CI]": [
            _fmt_estimate(row.curator_minus_audience_log_odds, row.difference_conf_low, row.difference_conf_high)
            for row in associations_scope.itertuples()
        ],
        "Gap $p$": associations_scope["difference_p_value"].map(_fmt_p_value),
        "Gap $q$": associations_scope["difference_q_value_bh"].map(_fmt_p_value),
    })
    coefficient_columns = ["Feature", "Unit", "Editor beta [95% CI]", "Editor $p$", "Editor $q$", "Audience beta [95% CI]", "Audience p", "Audience q", "Editor $-$ audience $\\Delta\\beta$ [95\\% CI]", "Gap $p$", "Gap $q$"]
    coefficient_table_flat = coefficient_table_flat[coefficient_columns]
    coefficient_table_flat.assign(scope=scope).to_csv(
        REGRESSION_TABLE_DIR / f"regression_coefficients_{scope}.csv", index=False
    )

    sectioned_rows = []
    section_titles = []
    for group, row in zip(associations_scope["term"].map(_regression_feature_group), coefficient_table_flat.to_dict("records"), strict=True):
        if group not in section_titles:
            section_titles.append(group)
            sectioned_rows.append({column: (FEATURE_GROUP_LABELS[group] if column == "Feature" else "") for column in coefficient_columns})
        sectioned_rows.append(row)
    coefficient_table = pd.DataFrame(sectioned_rows, columns=coefficient_columns)
    coefficient_table.columns = pd.MultiIndex.from_tuples([
        ("", "Feature"), ("", "Unit"),
        ("Editor", "β [95% CI]"), ("Editor", "p"), ("Editor", "q"),
        ("Audience", "β [95% CI]"), ("Audience", "p"), ("Audience", "q"),
        ("Gap (Editor-Audience)", "Δβ [95% CI]"), ("Gap (Editor-Audience)", "p"), ("Gap (Editor-Audience)", "q"),
    ])
    scope_coefficient_tables[scope] = coefficient_table
    all_coefficient_rows.append(coefficient_table_flat.assign(scope=scope))

    diagnostic_rows = []
    for key, label in diagnostic_order:
        value = diagnostic_overrides.get(key, diagnostics.get(key, sample.get(key)))
        if value is None:
            missing_regression_numbers.append({"scope": scope, "statistic": key})
            value = "MISSING — rerun 07_stacked_selection_models.Rmd"
        diagnostic_rows.append({"Diagnostic": label, "Value": _fmt_diagnostic(key, value), "statistic": key})
    diagnostic_table = pd.DataFrame(diagnostic_rows)[["Diagnostic", "Value"]]
    scope_diagnostic_tables[scope] = diagnostic_table
    pd.DataFrame(diagnostic_rows).to_csv(
        REGRESSION_TABLE_DIR / f"regression_diagnostics_{scope}.csv", index=False
    )
    all_diagnostic_rows.extend({"scope": scope, **row} for row in diagnostic_rows)

    coefficient_latex_table = coefficient_table.copy()
    coefficient_latex_table.columns = pd.MultiIndex.from_tuples([
        ("", "Feature"), ("", "Unit"),
        ("Editor", r"$\beta$ [95\% CI]"), ("Editor", "p"), ("Editor", "q"),
        ("Audience", r"$\beta$ [95\% CI]"), ("Audience", "p"), ("Audience", "q"),
        ("Gap (Editor-Audience)", r"$\Delta\beta$ [95\% CI]"), ("Gap (Editor-Audience)", "p"), ("Gap (Editor-Audience)", "q"),
    ])
    coefficient_latex = coefficient_latex_table.to_latex(
        index=False, escape=False, longtable=True, multicolumn=True, multicolumn_format="c",
        caption=f"Conditional-logit coefficient estimates: {scope}",
        label=f"tab:conditional-logit-coefficients-{scope}",
        column_format="@{}p{0.25\\linewidth}p{0.04\\linewidth}|p{0.12\\linewidth}p{0.04\\linewidth}p{0.04\\linewidth}|p{0.12\\linewidth}p{0.04\\linewidth}p{0.04\\linewidth}|p{0.12\\linewidth}p{0.04\\linewidth}p{0.04\\linewidth}@{}",
    )
    coefficient_latex = coefficient_latex.replace("\\multicolumn{2}{c}{}", "\\multicolumn{2}{c|}{}")
    for group, alignment in (("Editor", "c|"), ("Audience", "c|"), ("Gap", "c")):
        coefficient_latex = coefficient_latex.replace(
            f"\\multicolumn{{3}}{{c}}{{{group}}}",
            f"\\multicolumn{{3}}{{{alignment}}}{{{group}}}",
        )
    coefficient_latex = coefficient_latex.replace("\\midrule\n\\endfirsthead", "\\hline\n\\endfirsthead").replace("\\midrule\n\\endhead", "\\hline\n\\endhead")
    for group in section_titles:
        title = FEATURE_GROUP_LABELS[group]
        section_pattern = rf"^{re.escape(title)}(?:[ \t]&[ \t]*){{10}}\\\\$"
        coefficient_latex = _safe_re_sub(
            section_pattern,
            rf"\\multicolumn{{11}}{{l}}{{\\textit{{{title}}}}} \\\\\ ".rstrip(),
            coefficient_latex, flags=re.MULTILINE, count=1,
        )
    fixed_latex_lines = []
    for line in coefficient_latex.splitlines():
        if line.startswith("\\\\multicolumn{11}{l}{\\\\textit{"):
            title = line.split("{\\\\textit{", 1)[1].split("}}", 1)[0]
            fixed_latex_lines.append(
                r"\multicolumn{2}{l|}{\textit{" + title + r"}} & "
                r"\multicolumn{3}{l|}{} & "
                r"\multicolumn{3}{l|}{} & "
                r"\multicolumn{3}{l}{} " + chr(92) * 2
            )
        else:
            fixed_latex_lines.append(line)
    coefficient_latex = "\n".join(fixed_latex_lines)
    diagnostic_latex = diagnostic_table.to_latex(
        index=False, escape=True, longtable=True,
        caption=f"Conditional-logit diagnostics ({scope}); robust SEs clustered by article; held-out metrics use sealed test articles",
        label=f"tab:conditional-logit-diagnostics-{scope}",
        column_format="@{}p{0.65\\linewidth}p{0.25\\linewidth}@{}",
    )
    coefficient_html = coefficient_table.to_html(index=False, escape=True, classes="regression-table")
    for group in section_titles:
        title = FEATURE_GROUP_LABELS[group]
        section_html_pattern = (
            rf"    <tr>\n      <td>{re.escape(html_escape(title))}</td>\n"
            + rf"(?:      <td></td>\n){{10}}    </tr>"
        )
        coefficient_html = re.sub(
            section_html_pattern,
            f"    <tr class=\"feature-section\"><th colspan=\"2\">{html_escape(title)}</th><th colspan=\"3\"></th><th colspan=\"3\"></th><th colspan=\"3\"></th></tr>",
            coefficient_html, count=1,
        )
    latex_sections.extend([
        f"% ----- {scope} coefficient panel -----", coefficient_latex,
        f"% ----- {scope} fit and diagnostics panel -----", diagnostic_latex,
    ])
    html_sections.extend([
        f"<h3>{html_escape('All comments' if scope == 'all' else 'Root comments')}</h3>",
        "<h4>Coefficients, clustered SEs, 95% CIs, raw p-values, and BH q-values</h4>",
        coefficient_html,
        "<h4>Model fit and diagnostics</h4>",
        diagnostic_table.to_html(index=False, escape=True, classes="regression-diagnostics"),
    ])

(REGRESSION_TABLE_DIR / "regression_model_table.tex").write_text("\n\n".join(latex_sections) + "\n", encoding="utf-8")
(REGRESSION_TABLE_DIR / "regression_model_table.html").write_text(
    "<style>.regression-table,.regression-diagnostics{border-collapse:collapse;margin:0 0 1.5em;font-size:small}.regression-table th,.regression-table td,.regression-diagnostics th,.regression-diagnostics td{border:1px solid #bbb;padding:3px 5px}.regression-table th{white-space:nowrap}.regression-table td:first-child{white-space:nowrap}.regression-table thead tr:first-child th:nth-child(2),.regression-table thead tr:first-child th:nth-child(3),.regression-table thead tr:first-child th:nth-child(4),.regression-table thead tr:nth-child(2) th:nth-child(3),.regression-table thead tr:nth-child(2) th:nth-child(6),.regression-table thead tr:nth-child(2) th:nth-child(9),.regression-table tbody td:nth-child(3),.regression-table tbody td:nth-child(6),.regression-table tbody td:nth-child(9),.regression-table .feature-section th:nth-child(1),.regression-table .feature-section th:nth-child(2),.regression-table .feature-section th:nth-child(3){border-right:2px solid #555}.regression-table thead tr:first-child th:nth-child(2),.regression-table thead tr:first-child th:nth-child(3),.regression-table thead tr:first-child th:nth-child(4){text-align:center}.regression-table .feature-section th{border-top:2px solid #777;border-bottom:1px solid #777;text-align:left;font-style:italic;background:#f3f3f3}.regression-diagnostics th{white-space:nowrap}</style>" + "\n" + "\n".join(html_sections),
    encoding="utf-8",
)
pd.concat(all_coefficient_rows, ignore_index=True).to_csv(
    REGRESSION_TABLE_DIR / "regression_coefficients_full.csv", index=False
)
pd.DataFrame(all_diagnostic_rows).to_csv(
    REGRESSION_TABLE_DIR / "regression_diagnostics_full.csv", index=False
)
missing_regression_numbers = pd.DataFrame(missing_regression_numbers)
missing_regression_numbers.to_csv(
    REGRESSION_TABLE_DIR / "regression_missing_numbers.csv", index=False
)

print(f"LaTeX fragment: {REGRESSION_TABLE_DIR / 'regression_model_table.tex'}")
print(f"Rendered HTML: {REGRESSION_TABLE_DIR / 'regression_model_table.html'}")
if missing_regression_numbers.empty:
    print("All requested regression numbers are present.")
else:
    print("Missing regression numbers — rerun the stage-7 Rmd after the export update:")
    display(missing_regression_numbers)

display(HTML("\n".join(html_sections)))
